[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 03](README.md)

# OpenMP: bucles, reducciones y SIMD

**Tema:** 03 · **Sesiones:** 12, 13 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo distribuir iteraciones y combinar resultados sin introducir desbalance ni error numérico injustificado?


## Resultados de aprendizaje

- Comparar schedules con una carga conocida.
- Distinguir reduction, atomic y critical.
- Medir error numérico además del tiempo.


## Modelo conceptual

Una reducción crea acumuladores privados y una combinación definida por el operador.

Atomic protege una actualización compatible; critical serializa una región arbitraria.

La suma en punto flotante no es asociativa y el orden paralelo puede cambiar el redondeo.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "03"
NOTEBOOK = "03_openmp/02_bucles_reducciones.ipynb"
assert (ROOT / "curso" / "notebooks" / "03_openmp" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Balance de carga

Se compara una asignación estática contigua con round-robin para costos crecientes.


In [ ]:
costs = [1 + (i % 7) ** 2 for i in range(32)]
threads = 4
contiguous = [sum(costs[t*8:(t+1)*8]) for t in range(threads)]
cyclic = [sum(costs[t::threads]) for t in range(threads)]
def imbalance(loads): return max(loads) / (sum(loads) / len(loads))
assert sum(contiguous) == sum(costs) == sum(cyclic)
print("contiguo", contiguous, "desbalance", round(imbalance(contiguous), 3))
print("cíclico ", cyclic, "desbalance", round(imbalance(cyclic), 3))


**Interpretación.** El mejor schedule depende del costo por iteración, localidad y overhead; la tabla formula una hipótesis, no una regla universal.


## Precisión de la reducción

Se contrasta suma ingenua con `math.fsum` como referencia numérica más estable.


In [ ]:
import math
values = [1e16, 1.0, -1e16] * 1000
naive = sum(values)
stable = math.fsum(values)
print({"sum": naive, "fsum": stable, "error_absoluto": abs(naive - stable)})
assert stable == 1000.0


**Interpretación.** El resultado esperado y la tolerancia deben definirse antes de comparar estrategias paralelas.


## Práctica reproducible

1. Comparar `schedule(static)`, dinámico y guiado con la misma entrada.
2. Validar una integral/reducción frente a referencia.
3. Reportar tiempo, eficiencia y error para cada configuración.


## Errores frecuentes

- Usar critical para toda la iteración.
- Aceptar igualdad exacta de flotantes sin análisis.
- Vectorizar un bucle con dependencias.

## Criterios de aceptación

- Cláusula de reducción correcta.
- Tolerancia y referencia documentadas.
- Schedule y chunk registrados con el resultado.


## Referencias y material relacionado

- [Integral OpenMP](../../../openmp/integral.cc)
- [Reducción](../../../openmp/reduction/integral.cc)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 03](README.md)
